# Wildfire HTTP Evaluation + Final Artifacts

Use this notebook after GRPO training finishes. It keeps evaluation separate from training and drives the deployed OpenEnv Space over HTTP for the judge-facing showcase.

Run cells in order:
1. Pull the latest repo changes.
2. Generate reward/loss plots from `grpo_wildfire/log.jsonl`.
3. Run untrained baseline HTTP eval.
4. Run trained adapter HTTP eval.
5. Upload checkpoints/artifacts if needed.


In [ ]:
# Cell 1: repo update + constants
import os
import subprocess
import sys

BASE = "/home/user/app/wildfire-env"
SPACE_URL = "https://chunchunmaru-101-wildfire-env.hf.space"
os.chdir(BASE)

subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)
os.makedirs("submission_artifacts", exist_ok=True)
print("cwd:", os.getcwd())
print("OpenEnv HTTP base:", SPACE_URL)


In [ ]:
# Cell 2: generate training curves and summary
import subprocess
import sys

subprocess.run([sys.executable, "plot_training_curves.py"], check=True)
print("Wrote training plots and summary to submission_artifacts/")


In [ ]:
# Cell 3: untrained baseline eval through OpenEnv WebSocket session
# Uses official EnvClient on /ws for stateful reset/step + POST /grader for scoring.
# Episode length is governed by env max_steps (20 easy/medium, 25 hard).
import subprocess
import sys

subprocess.run([
    sys.executable,
    "eval_policy_http.py",
    "--untrained",
    "--base-url",
    SPACE_URL,
    "--seeds-per-task",
    "5",
    "--max-new-tokens",
    "128",
    "--output",
    "submission_artifacts/eval_untrained.json",
], check=True)
print("Wrote OpenEnv baseline: submission_artifacts/eval_untrained.json")


In [ ]:
# Cell 4: trained adapter eval through OpenEnv WebSocket session
import subprocess
import sys

subprocess.run([
    sys.executable,
    "eval_policy_http.py",
    "--base-url",
    SPACE_URL,
    "--seeds-per-task",
    "5",
    "--max-new-tokens",
    "128",
    "--output",
    "submission_artifacts/eval_trained.json",
], check=True)
print("Wrote OpenEnv trained eval: submission_artifacts/eval_trained.json")


In [ ]:
# Cell 5: upload checkpoints to HF model repo (optional but recommended)
import os
from huggingface_hub import HfApi, create_repo

HF_REPO_ID = os.environ.get("HF_REPO_ID", "Chunchunmaru-101/wildfire-grpo-checkpoints")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN before upload.")

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path="grpo_wildfire",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload wildfire GRPO checkpoints",
    ignore_patterns=["*.pyc", "__pycache__/*"],
)
print(f"Uploaded grpo_wildfire to https://huggingface.co/{HF_REPO_ID}")


In [ ]:
# Cell 6: quick artifact view
import json

for path in ["submission_artifacts/eval_untrained.json", "submission_artifacts/eval_trained.json"]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(path, "overall_mean=", data.get("overall_mean"))
